# Carrier fraction as a clamped PGM

The `homeostasis_telos` question, asked in THRML's language: **at what clamp density does intent percolate?**

- carriers → **clamped** `SpinNode`s pinned to the target
- intent diffusion → block Gibbs relaxation of the free spins
- the target moves → clamp values flip mid-run
- homeostatic score = mean edge agreement (target-blind) · teleological score = alignment with the moved target

Below the critical `h`, the lattice stays metastably locked in the old state after the flip — every local check green, globally wrong. The oracle found the Kuramoto threshold near `h ≈ 0.04`; here we look for its Gibbs-dynamics analog.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))  # works without pip install -e

import numpy as np
import matplotlib.pyplot as plt

from carrier_pgm import RingSpec, run_experiment, sweep, critical_h

## One run, watched closely

Phase A locks the lattice onto the old target; at the dashed line the purpose moves (clamps flip). Watch whether the teleological score follows.

In [ ]:
def watch(h, **kw):
    res = run_experiment(RingSpec(h=h), **kw)
    t_a = np.arange(len(res.hom_a))
    t_b = np.arange(len(res.hom_b)) + len(res.hom_a)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t_a, res.hom_a, c="tab:blue", label="homeostatic (edge agreement)")
    ax.plot(t_b, res.hom_b, c="tab:blue")
    ax.plot(t_a, res.tel_a, c="tab:red", label="teleological (vs current target)")
    ax.plot(t_b, res.tel_b, c="tab:red")
    ax.axvline(len(res.hom_a), ls="--", c="k", lw=1, label="the purpose moves")
    ax.set(xlabel="Gibbs sweep", ylabel="score", ylim=(-1.1, 1.1),
           title=f"h = {h:.2f}   (post-perturbation: hom={res.summary()[0]:.2f}, tel={res.summary()[1]:.2f}, gap={res.summary()[2]:.2f})")
    ax.legend(loc="lower left")
    plt.show()
    return res

_ = watch(0.0)   # pure homeostasis: expect FROZEN — locked, off-target

In [ ]:
_ = watch(0.10)  # seeded intent: do the clamps nucleate the flip?

## The h-sweep (the oracle's table)

In [ ]:
rows = sweep([0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 0.80, 1.0])

## Critical-threshold scan

Fine scan of `h`, looking for where teleological fidelity crosses 0.6 — the oracle's criterion.

In [ ]:
hs = np.linspace(0, 0.3, 16)
scan = sweep(hs, verbose=False)
tels = [r[2] for r in scan]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hs, tels, "o-", c="tab:red")
ax.axhline(0.6, ls=":", c="gray")
hc = critical_h(scan)
if hc is not None:
    ax.axvline(hc, ls="--", c="k", lw=1)
    ax.set_title(f"teleological fidelity crosses 0.6 near h = {hc:.3f}")
else:
    ax.set_title("no crossing in range — tune beta/j (metastability depth)")
ax.set(xlabel="carrier fraction h", ylabel="teleological fidelity (post-flip)")
plt.show()

## Knobs worth turning

`(beta, j)` set the depth of the metastable trap — the inverted analog of the oracle's noise. Too shallow: even `h=0` flips by fluctuation (no trap, no experiment). Too deep: nothing flips on your watch window. The threshold `h_c(beta, j)` is itself the interesting object — the oracle's single number becomes a phase boundary here.

Next steps sketched in the README: clock-model phases via `CategoricalNode` (the real 1.9-rad target shift), annealed β schedules, learned couplings via `IsingTrainingSpec`, and small-world rewiring of the ring.